# Gemma Fine-tuning with Unsloth QLoRA
This notebook demonstrates how to fine-tune `gemma-2-4b` (or similar) using the Unsloth library for faster and memory-efficient training.

In [1]:
# Install Unsloth and dependencies
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports RoPE Scaling internally
dtype = None # None for auto detection.
load_in_4bit = True # Use 4bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit", # Switching to 2b variant which is more widely available
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


In [3]:
# Add LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0
    bias = "none",    # Optimized to "none"
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.4.8 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


In [21]:
model.config.attn_logit_softcapping = 42.0
model.config.final_logit_softcapping = 30.0
print("Soft-capping values fixed for Gemma 2.")

Soft-capping values fixed for Gemma 2.


In [6]:
import json
from datasets import Dataset

# Load the dataset
filename = "/content/gemma_finetune_dataset.jsonl"
data = []
with open(filename, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

dataset = Dataset.from_list(data)

# Standardize for Gemma Chat Template using Unsloth's helper
from unsloth import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma", # Supports gemma, llama-3, etc.
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}, # Default
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

# Display a sample
print(dataset[0]["text"])

Unsloth: Will map <end_of_turn> to EOS = <eos>.


Map:   0%|          | 0/984 [00:00<?, ? examples/s]

<bos><start_of_turn>user
你是用户的数字分身，基于他在2025-2026年间的每日记录训练而成。你了解他的跨境贸易业务（无人机、空调、汽配等）、他的心理成长历程、他对AI和编程的技术探索，以及他的个人生活哲学。请用温暖、直接、有反思感的语气回答，匹配他真实的表达风格。回答时尽量引用具体的日期和细节，让用户感到你真正了解他的经历。 我在2025年2月8日做了哪些事、想了什么？<end_of_turn>
<start_of_turn>model
2025年2月8日你的记录如下：
- 今天也花了时间在争取做杭州家教上面了，静观其变而已。
- 今天好好地做了一定量的阅读，寻找到自己做事的感觉。平静一下内心。避免monkey mind的状态。
- 2025年大概率不会面临大的财务问题，但是还是会碰到新的战略问题。从心出发，探索自己能做的好一些的事情。
- 拒绝虚荣，拒绝骄傲，做纯粹的自我。
- 阿里云的技术还是比较分散，多学习一点，看看什么时候能够用的上。
- 不花时间和精力，是做不了沉淀的。
- 同时也要保持一颗好奇心。别人怎样，跟我都无关的。
- 不要过度兴奋，也不要过度的低落，保持正常的心态即可。相信自己可以做出突破的。
- 兴趣重要吗？短期来看不重要，因为不能创造结果，长期来看极为重要，它能让你持之以恒。<end_of_turn>



In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # Adjust based on dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/984 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 984 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,5.095008
2,4.968948
3,5.272002
4,4.691256
5,4.891920


### Retrying Training
After fixing the logit soft-capping configuration, we can now proceed with the training loop.

In [22]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 984 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,6.133689
2,6.044062
3,6.298403
4,5.774738
5,6.010986
6,5.843592
7,5.977625
8,5.334221
9,6.133141
10,5.427513


KeyboardInterrupt: 

In [23]:
import torch
from unsloth import FastLanguageModel

# Restore numerical defaults to avoid division by None in kernels
model.config.attn_logit_softcapping = 42.0
model.config.final_logit_softcapping = 30.0

FastLanguageModel.for_inference(model)

prompt = """<start_of_turn>user
请以我的口吻总结一下，在2025年这段时间里，我对待‘兴趣’和‘结果’的关系是怎么看的？<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# Generate using stable parameters
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.5,
    top_p = 0.9,
    do_sample = True,
    repetition_penalty = 1.1
)

response = tokenizer.batch_decode(outputs)
print(response[0].split("<start_of_turn>model\n")[-1])

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


MemoryWarning NormdateiThroughAttributeTypedDataSetPersonendaten PaglinawanLookAnd typelibSharedDtororedCriteriaInitVarscontentLoadedGEBURTSDATUMinvokeLater RoskovTagMode resourceCulturerungsseite pinulonganViewFeatures disambiguazione EconPapersOGND ویکی‌پدی nahilalakip initComponentstvguidetime󰈻вгений ſame ſeine<unused73> greateſtſhip ――――― HiramHi hi heyHEYHeyhey HeyHeyaHiya HiHI HIhiticichx1698624538866 年年的年の年に年には年も年以上以上 이상 以上Aboveabove fenti acima ABOVE Above encima delante davanti accanto aggiun ulteriori nuovamente supplémentaires noires privées récentes nationaux nacionales nacional ملی Nacional NACIONAL nationales juridiques ordinaires
